Fixing CNN build on [CNN](../pytorch/MODELS/simple_cnn.ipynb) using transfer learning

In [ ]:
import torch
import torch.nn as nn
from torchvision.datasets import Flowers102
import matplotlib.pyplot as plt
from torchvision import transforms
import numpy as np
from torch.utils.data import DataLoader
import torch.optim as optim 
import torchvision.models as models

In [ ]:
transform=transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_ds=Flowers102(root="data",split="train",download=True,transform=transform)
test_ds=Flowers102(root="data",split="test",download=True,transform=transform)

In [ ]:
train_loader=DataLoader(train_ds,batch_size=32,shuffle=True,pin_memory=True)
test_loader=DataLoader(test_ds,batch_size=32,shuffle=True,pin_memory=True)

In [ ]:
image,label=train_ds[37]
plt.imshow(image.permute(1, 2, 0))
plt.title(train_ds.classes[label])
plt.axis("off")
plt.show()

In [ ]:
resnet=models.resnet50(pretrained=True)

In [ ]:
resnet

In [ ]:
## we remove fc and make our own fc
resnet.fc

In [ ]:
"""we gonna 
Freeze: conv1, bn1, layer1, layer2, layer3
Train:  layer4, fc

"""

In [ ]:
## freeze all params
for param in resnet.parameters():
    param.requires_grad = False
### unfreeze only required one
for param in resnet.layer4.parameters():
    param.requires_grad = True

for param in resnet.fc.parameters():
    param.requires_grad = True

In [ ]:
!pip install torchviz

In [ ]:
from torchviz import make_dot
import torch

x = torch.randn(1, 3, 224, 224)
y = resnet(x)

dot = make_dot(y, params=dict(resnet.named_parameters()))


In [ ]:
resnet.fc=nn.Sequential(
nn.Linear(resnet.fc.in_features,512),
nn.ReLU(),
nn.Dropout(0.3),
nn.Linear(250,102)
)

In [ ]:
resnet.fc

In [ ]:
resnet =resnet.to(device=device)

In [ ]:
learning_rate=0.001
epoch=10
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(list(resnet.layer4.parameters()) +list(resnet.fc.parameters()),lr=learning_rate)

In [ ]:
model=resnet

In [ ]:
t=0
for i in range(epoch):
    resnet.train()
    running_loss=0
    for batch_x,batch_y in train_loader:
        print(f"Start of {t}th training ")
 ## just for checking loop        batch_x,batch_y=batch_x.to(device),batch_y.to(device)
        optimizer.zero_grad()
        pred=resnet(batch_x)
        loss=criterion(pred,batch_y)
        running_loss+=loss.item()
        loss.backward()
        optimizer.step()
        t=t+1
    print(f"Epoch {i+1}: Loss = {running_loss/len(train_loader):.4f}")    

In [ ]:
resnet.eval()

correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)

        outputs = resnet(x)
        preds = outputs.argmax(dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

print(f"Test Accuracy: {100*correct/total:.2f}%")